In [2]:
import numpy as np
import pandas as pd
import os
import gc

!pip install mlflow dagshub xgboost -q

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("DAGSHUB_USERNAME")

import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow")
mlflow.set_experiment("XGBoost_Training")
print("MLflow connected!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 77.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
train_t = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_i = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
train = train_t.merge(train_i, on="TransactionID", how="left")
del train_t, train_i
gc.collect()
print(f"Train: {train.shape}")

Train: (590540, 434)


In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_threshold=0.9):
        self.null_threshold = null_threshold
    
    def fit(self, X, y=None):
        X = X.copy()
        X.columns = [c.replace("id-", "id_") for c in X.columns]
        if "TransactionID" in X.columns:
            X = X.drop(columns=["TransactionID"])
        missing_pct = X.isnull().sum() / len(X)
        self.high_null_cols_ = missing_pct[missing_pct > self.null_threshold].index.tolist()
        X = X.drop(columns=self.high_null_cols_)
        self.num_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        X[self.num_cols_] = X[self.num_cols_].fillna(-999)
        X[self.cat_cols_] = X[self.cat_cols_].fillna("missing")
        self.label_encoders_ = {}
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(X[col].astype(str))
            self.label_encoders_[col] = le
        self.card1_counts_ = X["card1"].value_counts().to_dict()
        self.card1_amt_mean_ = X.groupby("card1")["TransactionAmt"].mean().to_dict()
        for col in self.cat_cols_:
            X[col] = self.label_encoders_[col].transform(X[col].astype(str))
        X = self._add_features(X)
        self.feature_columns_ = X.columns.tolist()
        return self
    
    def transform(self, X):
        X = X.copy()
        X.columns = [c.replace("id-", "id_") for c in X.columns]
        if "TransactionID" in X.columns:
            X = X.drop(columns=["TransactionID"])
        cols_to_drop = [c for c in self.high_null_cols_ if c in X.columns]
        X = X.drop(columns=cols_to_drop)
        for col in self.num_cols_:
            if col in X.columns:
                X[col] = X[col].fillna(-999)
        for col in self.cat_cols_:
            if col in X.columns:
                X[col] = X[col].fillna("missing")
        for col in self.cat_cols_:
            if col in X.columns:
                le = self.label_encoders_[col]
                known = set(le.classes_)
                fallback = le.classes_[0]
                X[col] = X[col].astype(str).apply(lambda v: v if v in known else fallback)
                X[col] = le.transform(X[col])
        X = self._add_features(X)
        for col in self.feature_columns_:
            if col not in X.columns:
                X[col] = 0
        X = X[self.feature_columns_]
        return X
    
    def _add_features(self, X):
        X["Transaction_hour"] = (X["TransactionDT"] / 3600) % 24
        X["Transaction_dow"] = (X["TransactionDT"] / 86400) % 7
        X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        X["TransactionAmt_decimal"] = (X["TransactionAmt"] - X["TransactionAmt"].astype(int)).round(2)
        X["Card1_count"] = X["card1"].map(self.card1_counts_).fillna(0)
        X["Card1_TransactionAmt_mean"] = X["card1"].map(self.card1_amt_mean_).fillna(X["TransactionAmt"].mean())
        X["Amt_div_card1mean"] = X["TransactionAmt"] / (X["Card1_TransactionAmt_mean"] + 1)
        return X

print("FraudPreprocessor defined")

FraudPreprocessor defined


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

y = train["isFraud"]
X = train.drop(columns=["isFraud"])

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
del train, X
gc.collect()
print(f"Train raw: {X_train_raw.shape}, Val raw: {X_val_raw.shape}")

Train raw: (472432, 433), Val raw: (118108, 433)


In [6]:
pipeline = Pipeline([
    ("preprocessor", FraudPreprocessor(null_threshold=0.9)),
    ("model", XGBClassifier(
        n_estimators=1000, max_depth=12, learning_rate=0.1,
        random_state=42, eval_metric="auc",
        tree_method="hist", device="cuda", n_jobs=-1,
    )),
])

print("Fitting Pipeline on raw data...")
pipeline.fit(X_train_raw, y_train)

val_pred = pipeline.predict_proba(X_val_raw)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print(f"Val AUC: {val_auc:.4f}")

with mlflow.start_run(run_name="XGBoost_Pipeline_FINAL") as run:
    mlflow.log_param("model", "XGBoost_Pipeline")
    mlflow.log_metric("val_auc", val_auc)
    mlflow.sklearn.log_model(
        pipeline,
        name="model",
        registered_model_name="ieee-fraud-best-model",
    )
    print("Pipeline registered as 'ieee-fraud-best-model'")

Fitting Pipeline on raw data...


/tmp/ipykernel_57/3867785139.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["Transaction_hour"] = (X["TransactionDT"] / 3600) % 24
/tmp/ipykernel_57/3867785139.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["Transaction_dow"] = (X["TransactionDT"] / 86400) % 7
/tmp/ipykernel_57/3867785139.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To g

Val AUC: 0.9750


2026/05/02 15:37:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ieee-fraud-best-model' already exists. Creating a new version of this model...
2026/05/02 15:38:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ieee-fraud-best-model, version 3
Created version '3' of model 'ieee-fraud-best-model'.


Pipeline registered as 'ieee-fraud-best-model'
🏃 View run XGBoost_Pipeline_FINAL at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/a8a9b5eab2634033800c38cb04fa5c3c
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
